In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
from random import randrange
import random
import numpy as np

In [25]:
bits = 16
LUT_BITS = 8
LUT_SIZE = 1 << LUT_BITS
F = bits - 1


lut = [
    int(((1 << (LUT_BITS + 2)) / ((1 << (LUT_BITS + 1)) + (2 * i + 1))) ** 0.5
        * (1 << F))
    for i in range(LUT_SIZE)
]

SQRT2_FP = int(math.sqrt(2) * (1 << F))
ONE_FP = 1 << F


a = random.getrandbits(bits)

parity = a.bit_length() & 1
half_bl = a.bit_length() >> 1  # = bl // 2, exact integer for both parities

shift = bits - a.bit_length()
a_hat = a << shift  # exactly `bits` bits, m = a_hat/2^bits in [0.5,1)

i = (a_hat >> (bits - 1 - LUT_BITS)) % 2**LUT_BITS
x = lut[i]  # x/2^F ~= 1/sqrt(m)

m = a_hat
for _ in range(bits // LUT_BITS):
    x2 = (x * x) >> F
    mx2 = (m * x2) >> bits
    x = (x * ((3 << F) - mx2)) >> (F + 1)
# x/2^F ~= 1/sqrt(m) at this point (this part is identical to before)

# Get sqrt(m) via a MULTIPLY instead of a divide:
#   y = m * x, then y/2^F ~= sqrt(m)   [since m/2^bits * (1/sqrt(m/2^bits))
#   = sqrt(m/2^bits), and the shift amounts (bits for m, F for x) net to F]
y = (m * x) >> bits  # y/2^F ~= sqrt(m/2^bits) = sqrt(a_hat/2^bits)

# sqrt(a) = sqrt(a_hat >> shift) = sqrt(a_hat / 2^shift)
#         = sqrt(a_hat) / 2^(shift/2)
# and sqrt(a_hat) = sqrt(m/2^bits) * 2^(bits/2) = (y/2^F) * 2^(bits/2)
# combining: sqrt(a) = y * 2^(bits/2 - shift/2 - F) = y * 2^(bl/2 - F)
#   [since bits - shift = bl]
# bl/2 may be a half-integer -> split as half_bl + (parity ? 0.5 : 0),
# and fold the 0.5 into the exact fixed-point sqrt(2) constant, exactly
# as before:
corr = SQRT2_FP * parity + ONE_FP * (1 - parity)  # 1 or sqrt(2), F frac bits

# result/1 ~= y * corr * 2^(half_bl - F) / 2^F   -- combine exponents:
# y has F frac bits, corr has F frac bits, so y*corr has 2F frac bits;
# we want an integer result scaled by 2^half_bl, i.e. shift right by 2F,
# then shift left by half_bl -- do the left shift first to preserve
# precision when half_bl > 2F is not the case here (half_bl <= 32, 2F=126),
# so shift right first is fine and avoids ever growing beyond need:
numerator = y * corr  # 2F fractional bits
# numerator >> (2F - half_bl) would divide if (2F - half_bl) were negative;
# since half_bl <= bits/2 = 32 and 2F = 2*63 = 126, (2F - half_bl) > 0 always
# for bits=64, so this is always a RIGHT shift -- no division:
result = numerator >> (2 * F - half_bl)

# Two O(1) last-ULP corrections (comparisons, not divisions or loops):
over = 1 if result * result > a else 0
result = result - over
under = 1 if (result + 1) * (result + 1) <= a else 0
result = result + under
 


print("Expected: {}".format(math.isqrt(a)))
print("Obtained: {}".format(result))

Expected: 239
Obtained: 239


In [16]:
lut[0].bit_length()

16

In [27]:
202079 % 2**16

5471

## Mapping the LUT to LUT_BITS polynomials

In [17]:
def get_bit(n, i):
    return (n >> i) & 1

In [18]:
def polylut(bit = 0):
    N = LUT_SIZE
    seq = np.array([get_bit(lut[i], bit) for i in range(N)], dtype=float)
    c = np.fft.fft(seq)  

    def f(x):
        s = 0.0
        for k in range(N):
            ang = 2 * math.pi * k * x / N
            s += c[k].real * math.cos(ang) - c[k].imag * math.sin(ang)
        return s / N

    return f

In [19]:
class Chebyshev:
    """
    Chebyshev(a, b, n, func)
    Given a function func, lower and upper limits of the interval [a,b],
    and maximum degree n, this class computes a Chebyshev approximation
    of the function.
    Method eval(x) yields the approximated function value.
    """
    def __init__(self, a, b, n, func):
        n = n + 1
        self.a = a
        self.b = b
        self.func = func

        bma = 0.5 * (b - a)
        bpa = 0.5 * (b + a)
        f = [func(math.cos(math.pi * (k + 0.5) / n) * bma + bpa) for k in range(n)]
        self.roots = f

        self.x = [math.cos(math.pi * (k + 0.5) / n) * bma + bpa for k in range(n)]

        fac = 2.0 / n
        self.c = [fac * sum([f[k] * math.cos(math.pi * j * (k + 0.5) / n)
                  for k in range(n)]) for j in range(n)]

    def eval(self, x):
        a,b = self.a, self.b
        #assert(a <= x <= b)
        y = (2.0 * x - a - b) * (1.0 / (b - a))
        y2 = 2.0 * y
        (d, dd) = (self.c[-1], 0)             # Special case first step for efficiency
        for cj in self.c[-2:0:-1]:            # Clenshaw's recurrence
            (d, dd) = (y2 * d - dd + cj, d)
        return y * d - dd + 0.5 * self.c[0]   # Last step is different

In [20]:
# Notice: for large bit sizes (64, 128, ...), this might take a few minutes

polyluts = []

for bit in range(bits + 2):
    f = polylut(bit)
    
    c = Chebyshev(0, LUT_SIZE, 851, f)
    
    polyluts.append(c)

In [21]:
# Exporting them 
if False:
    for i in range(len(polyluts)):
        with open("LUT-SQUARE-ROOT-{}-BITS-{}.txt".format(bits, i), "w") as f:
            f.write(", ".join(map(str, polyluts[i].c)))

---

### Now evaluating the division using polynomials only

In [32]:
bits = 16
LUT_BITS = 8
LUT_SIZE = 1 << LUT_BITS
F = bits - 1

SQRT2_FP = int(math.sqrt(2) * (1 << F))
ONE_FP = 1 << F


a = random.getrandbits(bits)
a = 26544

print("a", a)

bl = a.bit_length()
parity = bl & 1
half_bl = bl >> 1  

shift = bits - bl
a_hat = a << shift  

print("a_hat", a_hat)

i = (a_hat >> (bits - 1 - LUT_BITS)) % LUT_SIZE

print(i)

#####################################
###### Binary LUT evaluation ########
#####################################

binx = []

for j in range(bits + 2):
    if j >= len(polyluts):
        binx.append(0)
    else:
        binx.append(polyluts[j].eval(i))

#print(binx)
        
polydec = ([round(x) for x in list(binx)][::-1])
x = int("".join(map(str, polydec)), 2)

print("Output LUT: ", x)

#####################################
#####################################
#####################################

m = a_hat
for _ in range(math.ceil(math.log2(bits / LUT_BITS))):
    x2 = (x * x) >> F
    print("x2 ({}):".format(x2.bit_length()), x2)
    mx2 = (m * x2) >> bits
    print("mx2 factors: {}, {}".format(m , x2))
    print("mx2 ({}): ".format(mx2.bit_length()), mx2)
    term1 = (3 << F) - mx2
    print("term1 ({}): ".format(term1.bit_length()), term1)
    print("term1 % 128 ({}): ".format((term1 >> 1).bit_length()), term1 >> 1)

   
    term1_hi = term1 >> bits                      # 0 or 1
    print("T1 hi: {}".format(term1_hi * x))
    term1_lo = term1 - (term1_hi << bits)          # low 128 bits
    print("T1 lo ({}): {}".format(term1_lo.bit_length(), term1_lo))
    full = x * term1_lo                           # 128x128 -> up to 256 bits
    xprova = full >> bits                          # top half, this is (x*term1_lo) >> 128
    
    print("xpartial ({}): {}".format(xprova.bit_length(), xprova))
    xprova += x * term1_hi                        # term1_hi is 0 or 1, so this is 0 or x

    x = xprova

    #x = (x * term1)
    #x = x >> (F + 1)
    
    print("x ({}):".format(x.bit_length()), x)
    
    print("xfinal ({}):".format(x.bit_length()), x)

y = (m * x) >> bits 

print("y {}: {}".format(y.bit_length(), y))

print("Parity: ", parity)
corr = SQRT2_FP * parity + ONE_FP * (1 - parity)  # 1 or sqrt(2), F frac bits

print("CORR: ", corr)

numerator = y * corr 

print("NUMERATOR: ", numerator)

result = numerator << half_bl
print("BLIND ROT:", result)
result = result >> 2 * F

# Two O(1) last-ULP corrections (comparisons, not divisions or loops):
#over = 1 if result * result > a else 0
#result = result - over
#under = 1 if (result + 1) * (result + 1) <= a else 0
#result = result + under

print("Expected: {}".format(math.isqrt(a)))
print("Obtained: {}".format(result))

a 26544
a_hat 53088
158
Output LUT:  36418
x2 (16): 40474
mx2 factors: 53088, 40474
mx2 (16):  32786
term1 (16):  65518
term1 % 128 (15):  32759
T1 hi: 0
T1 lo (16): 65518
xpartial (16): 36407
x (16): 36407
xfinal (16): 36407
y 15: 29491
Parity:  1
CORR:  46340
NUMERATOR:  1366612940
BLIND ROT: 174926456320
Expected: 162
Obtained: 162


In [10]:
37799352727652188709946847655777218851710036856137294753937893504240567975936 % 2**bits

72115785578970093208298145711959048192

In [11]:
(200224471283358524579373278246662945582 * 289155554781221480656287356684007604091) >> bits

170141105454835102579024353418940843918

In [12]:
"[ 1 0 0 0 1 1 1 0 0 0 0 1 1 1 1 1 0 1 0 1 0 0 0 0 1 0 1 1 0 1 0 1 1 0 0 1 1 1 0 0 0 0 1 0 1 0 1 1 0 1 0 1 1 0 0 0 0 1 1 0 1 0 1 1 0 1 1 0 1 1 0 1 0 0 0 1 1 1 0 0 0 0 1 1 0 0 1 1 1 0 0 0 1 0 0 1 0 0 0 1 1 0 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 ]".replace(" ", ", ")

'[, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ]'

In [18]:
x = int("".join(map(str, [1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0][:-1])), 2)
x

96723442475760846702894925855899049263360